In [ ]:
import torch
from torch.utils import data
from torch import nn
from torchvision import datasets, transforms
import numpy as np
from torch.utils.data import Subset
k=5
lr=0.01
batch_size=64
epoch=10

In [ ]:
def try_gpu(i=0):
    if torch.cuda.device_count()>=i+1:
        return torch.device(f'cuda:{i}')
    return torch.device('cpu')
device=try_gpu()

In [ ]:
train_path = "/kaggle/input/datasets/ifigotin/imagenetmini-1000/imagenet-mini/train"
val_path = "/kaggle/input/datasets/ifigotin/imagenetmini-1000/imagenet-mini/val"
train_transform=transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])
val_transform=transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(), #Tensor中的像素值 = 原像素值 / 255 会自动计算    这样做主要是因为神经网络一般使用浮点数计算，而且较小、统一的数值范围通常更容易训练
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])
train_dataset = datasets.ImageFolder(
    root=train_path,
    transform=train_transform
)

test_dataset = datasets.ImageFolder(
    root=val_path,
    transform=val_transform
)
#print(train_dataset[0][0],train_dataset[0][1])
print(len(train_dataset))

34745


In [4]:
def get_net():
    net = nn.Sequential(
    # 这里使用一个11*11的更大窗口来捕捉对象。
    # 同时，步幅为4，以减少输出的高度和宽度。
    # 另外，输出通道的数目远大于LeNet
    nn.Conv2d(3, 96, kernel_size=11, stride=4, padding=1), nn.ReLU(),
    nn.MaxPool2d(kernel_size=3, stride=2),
    # 减小卷积窗口，使用填充为2来使得输入与输出的高和宽一致，且增大输出通道数
    nn.Conv2d(96, 256, kernel_size=5, padding=2), nn.ReLU(),
    nn.MaxPool2d(kernel_size=3, stride=2),
    # 使用三个连续的卷积层和较小的卷积窗口。
    # 除了最后的卷积层，输出通道的数量进一步增加。
    # 在前两个卷积层之后，汇聚层不用于减少输入的高度和宽度
    nn.Conv2d(256, 384, kernel_size=3, padding=1), nn.ReLU(),
    nn.Conv2d(384, 384, kernel_size=3, padding=1), nn.ReLU(),
    nn.Conv2d(384, 256, kernel_size=3, padding=1), nn.ReLU(),
    nn.MaxPool2d(kernel_size=3, stride=2),
    nn.Flatten(),
    # 这里，全连接层的输出数量是LeNet中的好几倍。使用dropout层来减轻过拟合
    nn.Linear(6400, 4096), nn.ReLU(),
    nn.Dropout(p=0.5),
    nn.Linear(4096, 4096), nn.ReLU(),
    nn.Dropout(p=0.5),
    # 最后是输出层。由于这里使用Fashion-MNIST，所以用类别数为10，而非论文中的1000
    nn.Linear(4096, 1000))
    return net

In [5]:
def get_k_fold_data(
    k: int,
    i: int,
    train_dataset,
    valid_dataset=None,
    seed: int = 42
):
    if k < 2:
        raise ValueError("k必须大于等于2")

    if not 0 <= i < k:
        raise ValueError(f"i必须在0到{k - 1}之间")

    # 如果训练集和验证集使用相同的transform  传valid_dataset的目的是要用valid里的valid_transform
    if valid_dataset is None:
        valid_dataset = train_dataset

    if len(train_dataset) != len(valid_dataset):
        raise ValueError("两个Dataset必须对应同一批样本")

    num = len(train_dataset)

    # 生成0到num-1的随机样本编号
    generator = torch.Generator().manual_seed(seed)
    indices = torch.randperm(
        num,
        generator=generator
    ).tolist()

    begin = i * num // k
    end = (i + 1) * num // k

    # 不再切分数据，而是切分编号
    valid_indices = indices[begin:end]
    train_indices = indices[:begin] + indices[end:]

    # 根据编号构造子数据集
    train_data = Subset(
        train_dataset,
        train_indices
    )

    valid_data = Subset(
        valid_dataset,
        valid_indices
    )

    return train_data, valid_data

In [6]:
loss=nn.CrossEntropyLoss()

In [7]:
def init(m):
    if isinstance(m,nn.Linear) or isinstance(m,nn.Conv2d):
        nn.init.xavier_uniform_(m.weight)

In [ ]:

for i in range(1): ##先五折里面抽一折看看情况
    avg_train_loss=0
    avg_valid_loss=0
    avg_train_acc=0
    avg_valid_acc=0
    net=get_net().to(device)
    net.apply(init)
    train_data,valid_data=get_k_fold_data(k,i,train_dataset)
    loader_config = {
    "batch_size": batch_size,
    "num_workers": 4, #多个cpu进程同时读取进程
    "pin_memory": True, #把准备好的batch放到CPU的“锁页内存”中。普通CPU内存中的数据复制到GPU时，通常还需要经过额外处理；锁页内存可以更直接、更快地传输到GPU。
    "persistent_workers": True, #每个epoch结束后，不关闭那4个读取数据的CPU进程。
    "prefetch_factor": 2  #每个worker提前准备2个batch。你有4个worker，所以最多提前准备：
}

    train_iter = data.DataLoader(
        train_data,
        shuffle=True,
        **loader_config
    )

    valid_iter = data.DataLoader(
        valid_data,
        shuffle=False,
        **loader_config
    )
    trainer=torch.optim.SGD(net.parameters(),lr=lr,momentum=0.9,weight_decay=5e-4)
    for j in range(epoch):
        net.train()
        train_acc=0
        train_loss=0.0
        train_count=0
        for x,y in train_iter:
            x = x.to(device, non_blocking=True) #复制数据时尽量不阻塞CPU
            y = y.to(device, non_blocking=True)
            trainer.zero_grad()
            logit = net(x)
            l = loss(logit, y)
            l.backward()
            trainer.step()
            train_count += y.size(0)
            train_loss += l.item() * y.size(0)
            train_acc += (logit.argmax(dim=1) == y).sum().item()
        print(
            f"第{i+1}折第{j+1}epoch，"
            f"训练loss：{train_loss/train_count:.5f}，"
            f"训练准确率：{train_acc/train_count:.5f}"
        )
        net.eval()
        
        test_acc=0
        
        test_loss=0.0
        
        test_count=0
        with torch.no_grad():
            # for x,y in train_iter:
            #     x = x.to(device, non_blocking=True)
            #     y = y.to(device, non_blocking=True)
            #     logit=net(x)
            #     train_count+=y.size(0)
            #     l=loss(logit,y)
            #     train_loss+=l.item()*y.size(0)
            #     train_acc+=(logit.argmax(dim=1)==y).sum().item()
            # #print(f"第{i+1}轮的训练loss：{train_loss/train_count:.5f},训练准确率：{train_acc/train_count}")
            # avg_train_loss+=train_loss/train_count
            # avg_train_acc+=train_acc/train_count
            for x,y in valid_iter:
                x = x.to(device, non_blocking=True)
                y = y.to(device, non_blocking=True)
                logit=net(x)
                test_count+=y.size(0)
                l=loss(logit,y)
                test_loss+=l.item()*y.size(0)
                test_acc+=(logit.argmax(dim=1)==y).sum().item()
            print(f"第{i+1}折的第{j+1}epoch测试loss：{test_loss/test_count:.5f},测试准确率：{test_acc/test_count}")
            avg_valid_loss+=test_loss/test_count
            avg_valid_acc+=test_acc/test_count
    print(f"第{i+1}折的train_loss:{avg_train_loss/epoch},train_acc:{avg_train_acc/epoch},valid_loss:{avg_valid_loss/epoch},valid_acc{avg_valid_acc/epoch}")
    
    


第1折第1epoch，训练loss：6.88802，训练准确率：0.00284
第1折的第1epoch测试loss：6.85419,测试准确率：0.003309828752338466
第1折第2epoch，训练loss：6.83553，训练准确率：0.00399
第1折的第2epoch测试loss：6.81543,测试准确率：0.0031659231544107066
第1折第3epoch，训练loss：6.77901，训练准确率：0.00486
第1折的第3epoch测试loss：6.75034,测试准确率：0.0048927903295438195
第1折第4epoch，训练loss：6.70775，训练准确率：0.00648
第1折的第4epoch测试loss：6.70711,测试准确率：0.004173262339905022
第1折第5epoch，训练loss：6.63531，训练准确率：0.00806
第1折的第5epoch测试loss：6.65516,测试准确率：0.007626996690171248
第1折第6epoch，训练loss：6.55429，训练准确率：0.01094
第1折的第6epoch测试loss：6.58444,测试准确率：0.009785580659087639
第1折第7epoch，训练loss：6.45089，训练准确率：0.01266
第1折的第7epoch测试loss：6.43273,测试准确率：0.012519787019715067
第1折第8epoch，训练loss：6.28693，训练准确率：0.01623
第1折的第8epoch测试loss：6.34209,测试准确率：0.019858972514030796
第1折第9epoch，训练loss：6.10478，训练准确率：0.02281
第1折的第9epoch测试loss：6.18224,测试准确率：0.024032234853935817
第1折第10epoch，训练loss：5.92628，训练准确率：0.02968
第1折的第10epoch测试loss：6.07045,测试准确率：0.02806159159591308
第1折的train_loss:0.0,train_acc:0.0,valid_loss:6.539417394786828,valid

In [ ]:
loader_config = {
    "batch_size": batch_size,
    "num_workers": 4,
    "pin_memory": True,
    "persistent_workers": True,
    "prefetch_factor": 2
}

In [10]:
net=get_net().to(device)
net.apply(init)
final_train_iter=data.DataLoader(train_dataset,shuffle=True,**loader_config)
trainer=torch.optim.SGD(net.parameters(),lr=lr,momentum=0.9,weight_decay=5e-4)
for i in range(epoch):
    net.train()
    for x,y in final_train_iter:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        trainer.zero_grad()
        l=loss(net(x),y)
        l.backward()
        trainer.step()
        

In [11]:
test_iter=data.DataLoader(
    test_dataset,
    shuffle=False,
    **loader_config
)
net.eval()
test_count=0
test_acc=0
test_loss=0.0
with torch.no_grad():
    for x,y in test_iter:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        logit=net(x)
        l=loss(logit,y)
        test_count+=y.size(0)
        test_acc+=(logit.argmax(dim=1)==y).sum().item()
        test_loss+=l.item()*y.size(0)
    print(f"测试损失：{test_loss/test_count:.5f},测试准确率：{test_acc/test_count:.5f}")
    
    
    

测试损失：5.84332,测试准确率：0.03645
